# Inter-Annotator Agreement (IAA) for YOLO Bounding-Box Labels
**SoyPod-571 dataset – IoU-based agreement between two annotators**

This notebook compares two independent annotation sets of the same images and reports:
- Precision, Recall, F1-score (agreement) and disagreement rate (1 − F1)
- Mean IoU of matched boxes
- Per-image pod-count agreement (MAE, Pearson r, exact-count match)

**Matching method:** for every image, boxes from the two annotators are matched one-to-one using greedy matching on IoU (highest IoU first). A pair counts as a match only if IoU ≥ the threshold (0.5 and 0.75 by default). Annotator 1 is the reference.

**Input:** two folders of YOLO `.txt` label files (`class x_center y_center width height`, normalized). Polygon lines (`class x1 y1 x2 y2 ...`) are converted to their enclosing box. Both sets must use the same image preprocessing (e.g., 640×640 stretch).

Roboflow adds a random suffix to file names (e.g., `IMG_0025_jpeg.rf.54e1...txt`), so files are paired by the original image name (the part before `_jpeg` / `_jpg` / `.rf.`).

## 1. Settings
Set the paths to the two label folders. Optionally set the image folder to draw example overlays.

In [ ]:
# ---- EDIT THESE PATHS ----
ANNOTATOR1_DIR = "A1"          # reference labels (first author)
ANNOTATOR2_DIR = "A2"          # independent labels (second author)
IMAGE_DIR      = None          # optional: folder with the 640x640 images, e.g. "test_dataset/train/images"

IOU_THRESHOLDS = [0.5, 0.75]
OUTPUT_CSV     = "iaa_per_image.csv"


### (Optional) Google Colab: upload two zip files
Run this cell only in Colab. It asks you to upload the two label zips and extracts them to `A1/` and `A2/`.

In [ ]:
# Uncomment to use in Google Colab
# from google.colab import files
# import zipfile, os
# for target in ["A1", "A2"]:
#     print(f"Upload the zip for {target}")
#     up = files.upload()
#     name = list(up.keys())[0]
#     with zipfile.ZipFile(name) as z:
#         z.extractall(target + "_raw")
# # Point the folders to the extracted label directories (adjust if nested)
# ANNOTATOR1_DIR = "A1_raw"
# ANNOTATOR2_DIR = "A2_raw"


## 2. Imports and helper functions

In [ ]:
import os
import re
import math
import glob
import pandas as pd


def find_label_files(folder):
    """Return all .txt label files under a folder (searches sub-folders), skipping README files."""
    files = glob.glob(os.path.join(folder, "**", "*.txt"), recursive=True)
    return [f for f in files if not os.path.basename(f).lower().startswith("readme")]


def image_key(path):
    """Original image name without Roboflow suffix, e.g. 'IMG_0025_jpeg.rf.abc.txt' -> 'IMG_0025'."""
    name = os.path.basename(path)
    name = re.split(r"\.rf\.", name)[0]
    name = re.sub(r"(_jpe?g|_png)+$", "", name, flags=re.IGNORECASE)
    return os.path.splitext(name)[0]


def read_boxes(path):
    """Read YOLO labels and return boxes as (x1, y1, x2, y2) in normalized coordinates."""
    boxes = []
    with open(path) as fh:
        for line in fh:
            parts = line.split()
            if len(parts) < 5:
                continue
            vals = [float(v) for v in parts[1:]]
            if len(vals) == 4:                     # standard YOLO box
                xc, yc, w, h = vals
                boxes.append((xc - w / 2, yc - h / 2, xc + w / 2, yc + h / 2))
            else:                                  # polygon -> enclosing box
                xs, ys = vals[0::2], vals[1::2]
                boxes.append((min(xs), min(ys), max(xs), max(ys)))
    return boxes


def iou(a, b):
    """Intersection over union of two boxes (x1, y1, x2, y2)."""
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def greedy_match(ref, other, thr):
    """One-to-one greedy matching on IoU. Returns list of (i_ref, j_other, iou)."""
    pairs = sorted(((iou(r, o), i, j) for i, r in enumerate(ref) for j, o in enumerate(other)),
                   reverse=True)
    used_r, used_o, matches = set(), set(), []
    for v, i, j in pairs:
        if v < thr:
            break
        if i in used_r or j in used_o:
            continue
        used_r.add(i)
        used_o.add(j)
        matches.append((i, j, v))
    return matches


## 3. Load and pair the label files

In [ ]:
files1 = {image_key(f): f for f in find_label_files(ANNOTATOR1_DIR)}
files2 = {image_key(f): f for f in find_label_files(ANNOTATOR2_DIR)}

common = sorted(set(files1) & set(files2))
only1 = sorted(set(files1) - set(files2))
only2 = sorted(set(files2) - set(files1))

print(f"Annotator 1 files: {len(files1)}")
print(f"Annotator 2 files: {len(files2)}")
print(f"Paired images:     {len(common)}")
if only1: print("Only in annotator 1:", only1)
if only2: print("Only in annotator 2:", only2)

boxes1 = {k: read_boxes(files1[k]) for k in common}
boxes2 = {k: read_boxes(files2[k]) for k in common}
print(f"Total boxes - annotator 1: {sum(map(len, boxes1.values()))}, annotator 2: {sum(map(len, boxes2.values()))}")


## 4. Independence check
If many files are byte-for-byte identical, or the mean IoU of matched boxes is ~1.00, the second set was probably copied/edited from the first and the result is **not** a valid inter-annotator agreement.

In [ ]:
identical = sum(open(files1[k], "rb").read() == open(files2[k], "rb").read() for k in common)
print(f"Identical label files: {identical} of {len(common)}")
if identical > 0.2 * len(common):
    print("WARNING: many files are identical - the annotations may not be independent.")


## 5. Compute agreement

In [ ]:
def evaluate(thr):
    tp = n1 = n2 = 0
    ious = []
    rows = []
    for k in common:
        a, b = boxes1[k], boxes2[k]
        m = greedy_match(a, b, thr)
        tp += len(m); n1 += len(a); n2 += len(b)
        ious += [v for _, _, v in m]
        rows.append({"image": k, "boxes_annotator1": len(a), "boxes_annotator2": len(b),
                     f"matched_iou{thr}": len(m)})
    precision = tp / n2 if n2 else 0.0
    recall = tp / n1 if n1 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    c1 = [len(boxes1[k]) for k in common]
    c2 = [len(boxes2[k]) for k in common]
    mae = sum(abs(x - y) for x, y in zip(c1, c2)) / len(common)
    m1, m2 = sum(c1) / len(c1), sum(c2) / len(c2)
    cov = sum((x - m1) * (y - m2) for x, y in zip(c1, c2))
    sd = math.sqrt(sum((x - m1) ** 2 for x in c1) * sum((y - m2) ** 2 for y in c2))
    summary = {
        "IoU threshold": thr,
        "Boxes, annotator 1 (reference)": n1,
        "Boxes, annotator 2": n2,
        "Matched boxes": tp,
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1-score (agreement)": round(f1, 4),
        "Disagreement rate (1 - F1) %": round((1 - f1) * 100, 2),
        "Mean IoU of matched boxes": round(sum(ious) / len(ious), 4) if ious else float("nan"),
        "Per-image count MAE": round(mae, 2),
        "Per-image count Pearson r": round(cov / sd, 4) if sd else float("nan"),
        "Exact count agreement %": round(100 * sum(x == y for x, y in zip(c1, c2)) / len(common), 2),
    }
    return summary, pd.DataFrame(rows), ious


results, per_image, all_ious = {}, None, {}
for thr in IOU_THRESHOLDS:
    summary, df, ious = evaluate(thr)
    results[thr] = summary
    all_ious[thr] = ious
    per_image = df if per_image is None else per_image.merge(df, on=["image", "boxes_annotator1", "boxes_annotator2"])

table = pd.DataFrame({f"IoU >= {t}": pd.Series(results[t], dtype=object) for t in IOU_THRESHOLDS})
table = table.drop(index="IoU threshold")
table


## 6. Per-image results
Images with the largest count difference are listed first. The full table is saved as CSV.

In [ ]:
per_image["count_difference"] = per_image["boxes_annotator2"] - per_image["boxes_annotator1"]
per_image = per_image.sort_values("count_difference", key=abs, ascending=False)
per_image.to_csv(OUTPUT_CSV, index=False)
print(f"Saved: {OUTPUT_CSV}")
per_image.head(10)


## 7. Plots

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# IoU distribution of matched boxes (lowest threshold)
t0 = IOU_THRESHOLDS[0]
axes[0].hist(all_ious[t0], bins=20, range=(t0, 1.0), color="#1f4e79", edgecolor="white")
axes[0].set_xlabel("IoU of matched boxes")
axes[0].set_ylabel("Number of box pairs")
axes[0].set_title(f"IoU distribution (IoU >= {t0})")

# Per-image count agreement
c1 = per_image["boxes_annotator1"]; c2 = per_image["boxes_annotator2"]
lim = max(c1.max(), c2.max()) + 2
axes[1].scatter(c1, c2, color="#1f4e79", alpha=0.7)
axes[1].plot([0, lim], [0, lim], "--", color="gray")
axes[1].set_xlim(0, lim); axes[1].set_ylim(0, lim)
axes[1].set_xlabel("Pods per image - annotator 1")
axes[1].set_ylabel("Pods per image - annotator 2")
axes[1].set_title("Per-image pod count agreement")

plt.tight_layout()
plt.savefig("iaa_plots.png", dpi=300)
plt.show()


## 8. (Optional) Visual check of one image
Green = annotator 1, red dashed = annotator 2. Requires `IMAGE_DIR`.

In [ ]:
if IMAGE_DIR:
    from PIL import Image
    import matplotlib.patches as patches

    k = per_image.iloc[0]["image"]          # image with the largest count difference
    img_path = next((p for p in glob.glob(os.path.join(IMAGE_DIR, "**", "*"), recursive=True)
                     if image_key(p) == k and p.lower().endswith((".jpg", ".jpeg", ".png"))), None)
    if img_path:
        img = Image.open(img_path); W, H = img.size
        fig, ax = plt.subplots(figsize=(7, 7)); ax.imshow(img); ax.axis("off")
        for (x1, y1, x2, y2) in boxes1[k]:
            ax.add_patch(patches.Rectangle((x1 * W, y1 * H), (x2 - x1) * W, (y2 - y1) * H,
                                           fill=False, edgecolor="lime", linewidth=1.5))
        for (x1, y1, x2, y2) in boxes2[k]:
            ax.add_patch(patches.Rectangle((x1 * W, y1 * H), (x2 - x1) * W, (y2 - y1) * H,
                                           fill=False, edgecolor="red", linewidth=1.2, linestyle="--"))
        ax.set_title(f"{k}: annotator 1 = {len(boxes1[k])}, annotator 2 = {len(boxes2[k])}")
        plt.show()
    else:
        print("Image not found for", k)
else:
    print("Set IMAGE_DIR to draw an overlay.")


## 9. Sentence for the paper
Prints a ready-to-use sentence with the computed values.

In [ ]:
r5 = results[IOU_THRESHOLDS[0]]
r75 = results[IOU_THRESHOLDS[-1]]
print(
    f"At an IoU threshold of {IOU_THRESHOLDS[0]}, {r5['Matched boxes']:,} of the {r5['Boxes, annotator 1 (reference)']:,} "
    f"reference boxes and of the {r5['Boxes, annotator 2']:,} second-annotator boxes were matched, giving a precision of "
    f"{r5['Precision']:.4f}, a recall of {r5['Recall']:.4f}, an F1-score of {r5['F1-score (agreement)']:.4f}, and a "
    f"disagreement rate of {r5['Disagreement rate (1 - F1) %']:.2f}%. The mean IoU of matched boxes was "
    f"{r5['Mean IoU of matched boxes']:.4f}, and per-image pod counts were highly consistent (MAE "
    f"{r5['Per-image count MAE']:.2f} pods; Pearson r = {r5['Per-image count Pearson r']:.4f}). At the stricter IoU "
    f"threshold of {IOU_THRESHOLDS[-1]}, the F1-score was {r75['F1-score (agreement)']:.4f}."
)
